In [195]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [196]:
# Linear algebra
import numpy as np

# Data manipulation
import pandas as pd

# Predictive data analysis
from sklearn.compose import make_column_transformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, make_scorer, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder

In [197]:
# Tarining data set, used to build the model
train_data = pd.read_csv('../input/titanic/train.csv')

# Test data set, used to evaluate how well the model performs on unseen data 
test_data = pd.read_csv('../input/titanic/test.csv')

In [198]:
train_data.describe()

In [199]:
train_data.head(20)

In [200]:
# Drop unused features
train_data.drop(['Name', 'Cabin', 'Ticket'], axis=1, inplace=True)
test_data.drop(['Name', 'Cabin', 'Ticket'], axis=1, inplace=True)

In [201]:
# Number of NA's per variable in the train data set
train_data.isna().sum().sort_values()

In [202]:
# Number of NA's per variable in the test data set
test_data.isna().sum().sort_values()

In [203]:
# Impute missing values
train_data['Age'].fillna(train_data['Age'].mean(), inplace=True)
train_data.dropna(subset=['Embarked'], inplace=True)

test_data['Age'].fillna(test_data['Age'].mean(), inplace=True)
test_data['Fare'].fillna(test_data['Fare'].mean(), inplace=True)

In [204]:
# One-hot encoding
train_data = pd.get_dummies(train_data, columns=['Sex', 'Embarked'])
test_data = pd.get_dummies(test_data, columns=['Sex', 'Embarked'])

In [205]:
# Extract the independent feature matrix and the dependant vector from train
X_train = train_data.loc[:,train_data.columns != 'Survived']
y_train = train_data.iloc[:,1]

In [206]:
# Split data into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=1   
)

In [207]:
# Define the model
model = GradientBoostingClassifier(
    n_estimators=32, 
    learning_rate=0.1,   
    max_depth=4, max_features=6,                               
    min_samples_leaf=15, 
    min_samples_split=10, 
    random_state =5
)

In [208]:
# Train the model
model.fit(X_train, y_train)

In [209]:
# Make predictions using the validation dataset
val_predictions = model.predict(X_val)

In [210]:
# Confusion matrix
confusion_matrix(y_val, val_predictions)

In [211]:
# Classification report
classification_report(y_val, val_predictions)

In [212]:
# Make predictions using the test dataset
test_predictions = model.predict(test_data)

In [213]:
# Generate submission
submission = pd.DataFrame({'PassengerId':test_data['PassengerId'],'Survived':test_predictions})
submission.to_csv('titanic_gbc_predictions.csv',index=False)